# Etapa 4 · Fine-tuning do BERTimbau (Colab, GPU T4)

**Como abrir sem ocupar espaço no Drive:** acesse
`https://colab.research.google.com/github/thmlazaro/tp2-ia-2026.2/blob/main/notebooks/02_bertimbau.ipynb`
(o repositório precisa estar público). Se o Colab perguntar se quer salvar uma cópia, **não** é necessário.

**Antes de rodar:** menu **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4 → Salvar**.

O que este notebook faz:
1. **Parte 1:** treina `neuralmind/bert-base-portuguese-cased` para prever as 5 competências (regressão, rótulos ÷ 200) e escolhe a melhor época pelo **QWK médio no dev**.
2. **Parte 2 (numa sessão nova):** recarrega o modelo salvo e gera `preds_bertimbau.csv` para o teste.
3. **Download:** baixa as previsões, o histórico do treino, a figura e o modelo compactado.

Por padrão (`USAR_DRIVE = False`) tudo fica no disco do Colab, que é **apagado quando a sessão termina**: faça os downloads do final antes de fechar.

Todo o código está em `src/bertimbau.py` (o mesmo que o app usa). O teste **não** é usado para escolher nada: as métricas no teste são calculadas depois, pelo `src/avaliacao.py`.

Tempo esperado: ~20 a 40 min de treino na T4 (4 épocas, 4.592 redações).

## Parte 1 · Treino

In [ ]:
import torch
assert torch.cuda.is_available(), "Sem GPU! Ambiente de execução -> Alterar tipo -> T4"
print(torch.cuda.get_device_name(0))

In [ ]:
# Configuração (rode sempre no início de cada sessão)
import os, sys, subprocess
from pathlib import Path

USAR_DRIVE = False   # True só se o seu Google Drive tiver pelo menos ~1 GB livre

if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/TP2-IA")   # sobrevive ao fim da sessão
else:
    BASE = Path("/content/saida")                  # disco do Colab: baixe tudo ao final!
for sub in ["modelos", "results/figuras"]:
    (BASE / sub).mkdir(parents=True, exist_ok=True)

REPO = Path("/content/repo")
URL = "github.com/thmlazaro/tp2-ia-2026.2.git"
if not REPO.exists():
    url = f"https://{URL}"                         # repositório público
    try:
        from google.colab import userdata          # repositório privado: token nos Secrets
        url = f"https://{userdata.get('GITHUB_TOKEN')}@{URL}"
    except Exception:
        pass
    subprocess.run(["git", "clone", "-q", url, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=True)
sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)
print("código e dados em", REPO, "| saídas em", BASE)

In [ ]:
!pip -q install datasets accelerate   # já vêm no Colab na maioria das vezes; garante que existem
import transformers, datasets
print("transformers", transformers.__version__, "| datasets", datasets.__version__)

In [ ]:
import pandas as pd
train = pd.read_csv("data/train.csv")
dev = pd.read_csv("data/dev.csv")
print("treino:", len(train), "| dev:", len(dev))

Hiperparâmetros (ponto de partida do guia): lr 2e-5, 4 épocas, batch 8, fp16, weight decay 0,01, warmup 10%, semente 42, max_length 512.
Os checkpoints de cada época ficam em `/content/checkpoints`; o **melhor** modelo vai para `BASE/modelos/bertimbau_final`.

In [ ]:
import bertimbau
DIR_MODELO = BASE / "modelos" / "bertimbau_final"
info = bertimbau.treinar(train, dev, DIR_MODELO, "/content/checkpoints")
print("melhor QWK médio no dev:", round(info["melhor_qwk_medio_dev"], 4))
print("duração do treino (min):", round(info["duracao_treino_s"] / 60, 1))

In [ ]:
# Métricas no dev a cada época (vão para o relatório)
ev = pd.DataFrame([h for h in info["historico"] if "eval_qwk_medio" in h])
cols = ["epoch", "eval_loss", "eval_qwk_medio", "eval_mae_medio"] + [f"eval_qwk_c{k}" for k in range(1, 6)]
ev[cols].round(4)

In [ ]:
import shutil
fig = bertimbau.grafico_treino(info, BASE / "results" / "figuras" / "bertimbau_treino.png")
shutil.copy(DIR_MODELO / "treino.json", BASE / "results" / "bertimbau_treino.json")
shutil.rmtree("/content/checkpoints", ignore_errors=True)   # libera disco; o melhor modelo já foi salvo
print(sorted(p.name for p in DIR_MODELO.iterdir()))

## Parte 2 · Previsão do teste numa sessão NOVA

Isso prova que o modelo salvo carrega sozinho, sem nada da memória do treino (critério de "pronto" da etapa).

1. Menu **Ambiente de execução → Reiniciar sessão** (os arquivos em disco são mantidos).
   ⚠️ **Não** use "Desconectar e excluir ambiente de execução": isso apaga o modelo.
2. Rode a **célula de configuração** de novo.
3. Rode as células abaixo.

In [ ]:
import pandas as pd, bertimbau
DIR_MODELO = BASE / "modelos" / "bertimbau_final"
test = pd.read_csv("data/test.csv")
preds = bertimbau.prever_dataframe(test, DIR_MODELO)
preds.to_csv(BASE / "results" / "preds_bertimbau.csv", index=False)
assert len(preds) == len(test) and preds["id"].equals(test["id"])
assert preds[["c1", "c2", "c3", "c4", "c5"]].isin([0, 40, 80, 120, 160, 200]).all().all()
print("previsões:", len(preds))
preds.head()

In [ ]:
# Distribuição das notas previstas (sanidade; as métricas vêm depois, no avaliacao.py)
preds[["c1", "c2", "c3", "c4", "c5"]].apply(lambda s: s.value_counts()).fillna(0).astype(int)

In [ ]:
# A função que o app usa
print(bertimbau.predict(test["texto"].iloc[0], DIR_MODELO))

## Download (faça antes de fechar o Colab)

1. Rode a célula abaixo: ela baixa os 3 arquivos de resultado. Coloque `preds_bertimbau.csv` e `bertimbau_treino.json` em `results/` e a figura em `results/figuras/` no repositório.
2. A célula seguinte compacta o modelo (~400 MB) e baixa `bertimbau_final.zip`. Descompacte em `modelos/bertimbau_final/` no repositório (essa pasta fica fora do Git): o app precisa dela.

Se o navegador bloquear vários downloads, permita downloads múltiplos para o site do Colab, ou baixe pelo painel **Arquivos** (ícone de pasta à esquerda → `/content/saida` → botão direito → Fazer download).

In [ ]:
from google.colab import files
for f in ["preds_bertimbau.csv", "bertimbau_treino.json", "figuras/bertimbau_treino.png"]:
    files.download(str(BASE / "results" / f))

In [ ]:
import shutil
zip_modelo = shutil.make_archive(str(BASE / "bertimbau_final"), "zip", DIR_MODELO)
print(zip_modelo, round(Path(zip_modelo).stat().st_size / 1e6), "MB")
files.download(zip_modelo)